In [ ]:
import os
model_name = os.getenv("model_name")

In [6]:
import os
from prompts import System_message, user_input, assistant_message_role, System_message,assistant_message_zero_shot, assistant_message_one_shot, assistant_message_few_shot, assistant_message_self_consistency

prompt_tamplate = [
    {"role": "system", "content": System_message},
    {"role": "assistant", "content": assistant_message_role},
    {"role": "assistant", "content": assistant_message_zero_shot},
    {"role": "assistant", "content": assistant_message_one_shot},
    {"role": "assistant", "content": assistant_message_few_shot},
    {"role": "assistant", "content": assistant_message_self_consistency},
    {"role": "user", "content": user_input}
]

In [9]:
from groq_client import client
def call_groq_model(promtps, model="llama-3.3-70b-versatile", temperature=0.3):
    response = client.chat.completions.create(
        model=model,
        messages=promtps,
        temperature=temperature,
    )
    return response.choices[0].message.content


In [8]:
import json
OUTPUT_FOLDER = "Outputs"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

from groq_client import client
def call_groq_model(promtps, model="llama-3.3-70b-versatile", temperature=0.3):
    response = client.chat.completions.create(
        model=model,
        messages=promtps,
        temperature=temperature,
    )
    return response.choices[0].message.content


def save_json_output(filename="sample_blog_output.json"):
    """
    Save LLM JSON response into outputs folder.
    """
    try:
        llm_output = call_groq_model(prompt_tamplate)
        print(f"LLM Response:\n{llm_output}\n")
        import re

        
        try:
            json_data = json.loads(llm_output)
        except json.JSONDecodeError:
            
            cleaned = re.sub(r'```(?:json)?\\n', '', llm_output)
            cleaned = cleaned.replace('```', '').strip()

            
            try:
                json_data = json.loads(cleaned)
            except json.JSONDecodeError:
                
                m = re.search(r'(\\{.*\\}|\\[.*\\])', cleaned, re.S)
                if m:
                    candidate = m.group(1)
                    try:
                        json_data = json.loads(candidate)
                    except json.JSONDecodeError as e2:
                        print("✗ Invalid JSON after extraction.")
                        print(f"Error: {e2}")
                        print(f"Actual response (start): {llm_output[:1000]}")
                        return False
                else:
                    print("✗ No JSON-like substring found in the LLM response.")
                    print(f"Actual response (start): {llm_output[:1000]}")
                    return False

        # Save parsed JSON to file
        file_path = os.path.join(OUTPUT_FOLDER, filename)
        with open(file_path, "w", encoding="utf-8") as file:
            json.dump(json_data, file, indent=4, ensure_ascii=False)

        print(f"✓ JSON saved successfully at: {file_path}")
        return True

    except Exception as e:
        print(f"✗ Error occurred: {type(e).__name__}: {str(e)}")
        return False


save_json_output()


LLM Response:
```
{
  "blog_intent_analysis": {
    "blog_intent": "PRODUCT_EDUCATION",
    "target_reader_maturity": "INTERMEDIATE",
    "recommended_content_angle": "Practical applications of AI in customer support",
    "reasoning_summary": "The blog aims to educate customer support leaders and operations heads on the benefits of using AI in customer support operations, highlighting the features and advantages of the AI support assistant product."
  },
  "input_summary": {
    "topic": "AI in customer support",
    "target_audience": "Customer support leaders and operations heads",
    "product_service": "AI support assistant",
    "key_points": [
      "Reduces repetitive ticket handling",
      "Summarizes customer issues",
      "Suggests replies to support agents",
      "Helps improve response time",
      "Should not replace human agents completely"
    ],
    "tone": "Professional and practical",
    "blog_length": 900,
    "seo_keywords": [
      "AI customer support",
     

True